# Clinical NLP Extractor with Risk Assessment

This notebook demonstrates a clinical text extraction pipeline using:
- **Regex** for structured patterns (dates, ICD codes, lab values)
- **spaCy** for NLP entity tagging (diagnoses, drugs, symptoms)
- **LLMs** for flexible extraction and summarization

## Workflow
1. **Input:** Clinical notes (free text)
2. **Regex:** Identify ICD codes, dates, and numeric values
3. **spaCy:** Tag medical entities
4. **LLM:** Summarize and extract contextual information
5. **Risk Assessment:** Detect bias, check privacy compliance, and generate audit trails
6. **Output:** Structured JSON + Risk Assessment Report

In [ ]:
import re
import json
import sys
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Tuple

# Add parent directory to path to import medcoding module
sys.path.insert(0, str(Path.cwd()))

# Import our clinical extractor
from medcoding import ClinicalNLPExtractor, process_clinical_note

# Data processing libraries
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully")
print(f"✓ Current working directory: {Path.cwd()}")

In [ ]:
# Sample clinical notes for demonstration
sample_clinical_notes = [
    {
        'note_id': 'NOTE001',
        'date': '2024-01-15',
        'text': """
        Patient: John Doe
        Date: 2024-01-15
        Age: 45, Male
        
        Chief Complaint: Persistent cough and shortness of breath
        
        History of Present Illness:
        45-year-old male with type 2 diabetes (ICD-10: E11.65) and hypertension (ICD-10: I10)
        presents with fever of 102°F, productive cough for 2 weeks, and dyspnea on exertion.
        Patient reports chest pain and fatigue. Denies hemoptysis.
        
        Medical History:
        - Diabetes Mellitus Type 2 (E11.65)
        - Hypertension (I10)
        - Hyperlipidemia
        
        Current Medications:
        - Metformin 500mg twice daily
        - Lisinopril 10mg once daily
        - Atorvastatin 20mg at bedtime
        - Aspirin 81mg daily
        - Albuterol inhaler as needed
        
        Recent Lab Values:
        - Blood Glucose: 185 mg/dL
        - Systolic BP: 145 mmHg
        - WBC: 12.5 units
        - CRP: 8.2 mg/dL
        
        Assessment:
        1. Community-acquired pneumonia (ICD-10: J18.9)
        2. Type 2 Diabetes Mellitus with complications
        3. Hypertension, controlled
        
        Plan:
        - Start Amoxicillin 500mg three times daily for 7 days
        - Chest X-ray to rule out complications
        - Monitor blood glucose closely
        - Follow-up in 1 week
        """
    },
    {
        'note_id': 'NOTE002',
        'date': '2024-01-16',
        'text': """
        Patient: Jane Smith
        Date: 2024-01-16
        Age: 52, Female
        
        Chief Complaint: Follow-up for depression and anxiety
        
        History of Present Illness:
        52-year-old female with major depressive disorder and generalized anxiety disorder
        reports improved mood on current regimen. Sleep quality improving.
        
        Current Medications:
        - Sertraline 100mg daily
        - Alprazolam 0.5mg twice daily as needed
        - Vitamin B12 supplement
        
        Lab Values (Jan 10, 2024):
        - Thyroid TSH: 2.1 mIU/L (normal)
        - Vitamin B12: 450 pg/mL (normal)
        
        Assessment:
        Major depressive disorder (ICD-10: F32.9) - improving
        Generalized anxiety disorder (ICD-10: F41.1) - stable
        
        Plan:
        Continue current medications
        Psychotherapy weekly
        Recheck labs in 3 months
        """
    }
]

# Create DataFrame from sample notes
df = pd.DataFrame(sample_clinical_notes)
print(f"✓ Loaded {len(df)} clinical notes")
print(f"\nNotes Summary:")
print(df[['note_id', 'date']].to_string())
print(f"\nAverage note length: {df['text'].str.len().mean():.0f} characters")

In [ ]:
# Initialize the extractor
extractor = ClinicalNLPExtractor()

# Process first clinical note with regex extraction
first_note = sample_clinical_notes[0]['text']
print("="*60)
print("REGEX PATTERN EXTRACTION - First Note")
print("="*60)

regex_results = extractor.extract_regex_patterns(first_note)

print("\n📋 Extracted Structured Data:")
print(f"  • ICD Codes found: {regex_results['icd_codes']}")
print(f"  • Dates found: {regex_results['dates']}")
print(f"  • Lab Values: {regex_results['lab_values']}")
print(f"  • Dosages: {regex_results['dosages']}")

# Summary table
extraction_summary = pd.DataFrame({
    'Pattern Type': list(regex_results.keys()),
    'Count': [len(v) for v in regex_results.values()],
    'Examples': [
        ', '.join(regex_results[k][:2]) if regex_results[k] else 'None'
        for k in regex_results.keys()
    ]
})
print("\n" + extraction_summary.to_string(index=False))

In [ ]:
print("="*60)
print("spaCy NLP ENTITY EXTRACTION - First Note")
print("="*60)

# Extract spaCy entities
spacy_entities = extractor.extract_spacy_entities(first_note)

if spacy_entities:
    print("\n🏷️  Named Entities Identified:")
    for entity_type, entities in spacy_entities.items():
        print(f"\n  {entity_type}:")
        for ent in entities[:5]:  # Show first 5
            print(f"    - {ent['text']}")
        if len(entities) > 5:
            print(f"    ... and {len(entities) - 5} more")
else:
    print("\n⚠️  spaCy entities not extracted (model may not be installed)")
    print("Install with: python -m spacy download en_core_web_sm")

# Extract clinical concepts using keyword matching
clinical_concepts = extractor.extract_clinical_concepts(first_note)

print("\n\n📚 Clinical Concepts Identified:")
for concept_type, concepts in clinical_concepts.items():
    print(f"\n  {concept_type.title()}:")
    for concept in concepts:
        print(f"    ✓ {concept}")

In [ ]:
print("="*60)
print("LLM-BASED EXTRACTION SIMULATION")
print("="*60)

# Simulate LLM-based extraction with rule-based logic
def simulate_llm_extraction(note_text: str) -> Dict[str, Any]:
    """
    Simulate LLM extraction capabilities.
    In production, this would call OpenAI, Claude, or open-source LLM APIs.
    """
    extraction = {
        'chief_complaint': 'Persistent cough and shortness of breath',
        'key_findings': [
            'Fever 102°F',
            'Dyspnea on exertion',
            'Productive cough for 2 weeks',
            'Chest pain'
        ],
        'risk_factors': [
            'Pre-existing diabetes',
            'Hypertension',
            'Age 45'
        ],
        'primary_diagnosis': 'Community-acquired pneumonia (J18.9)',
        'secondary_diagnoses': [
            'Type 2 Diabetes Mellitus with complications (E11.65)',
            'Hypertension (I10)'
        ],
        'recommended_actions': [
            'Initiate antibiotic therapy',
            'Chest imaging to rule out complications',
            'Monitor glucose levels closely',
            '1-week follow-up'
        ],
        'clinical_summary': 'Middle-aged male with multiple comorbidities presenting with acute pneumonia. Requires close monitoring due to diabetes and hypertension complications.',
        'confidence_scores': {
            'primary_diagnosis': 0.92,
            'key_findings': 0.88,
            'risk_assessment': 0.85
        }
    }
    return extraction

# Simulate LLM extraction
llm_results = simulate_llm_extraction(first_note)

print("\n🤖 LLM-Enhanced Extraction Results:")
print(f"\n  Chief Complaint: {llm_results['chief_complaint']}")
print(f"\n  Primary Diagnosis: {llm_results['primary_diagnosis']} (Confidence: {llm_results['confidence_scores']['primary_diagnosis']:.0%})")
print(f"\n  Secondary Diagnoses:")
for diag in llm_results['secondary_diagnoses']:
    print(f"    • {diag}")

print(f"\n  Key Findings:")
for finding in llm_results['key_findings']:
    print(f"    • {finding}")

print(f"\n  Risk Factors:")
for risk in llm_results['risk_factors']:
    print(f"    ⚠️  {risk}")

print(f"\n  Clinical Summary:")
print(f"    {llm_results['clinical_summary']}")

print(f"\n  Recommended Actions:")
for action in llm_results['recommended_actions']:
    print(f"    → {action}")

In [ ]:
print("="*60)
print("RISK ASSESSMENT AND BIAS DETECTION")
print("="*60)

# Run complete extraction pipeline
extraction_results = extractor.extract_all(first_note)

# Generate risk assessment
risk_assessment = extractor.generate_risk_assessment(extraction_results)

print("\n📋 Data Privacy Assessment:")
privacy = risk_assessment['data_privacy']
print(f"  • PII Detection: {'⚠️  FOUND' if privacy['has_pii'] else '✓ CLEAR'}")
print(f"  • Status: {privacy['compliance_status']}")
print(f"  • Recommendation: {privacy['recommendation']}")

print("\n🎯 Bias Detection:")
bias = risk_assessment['bias_check']
print(f"  • Bias Detected: {'⚠️  YES' if bias['potential_bias_detected'] else '✓ NO'}")
print(f"  • Demographic Terms: {bias['demographic_terms_found'] if bias['demographic_terms_found'] else 'None found'}")
print(f"  • Recommendation: {bias['recommendation']}")

print("\n✅ Compliance Check:")
compliance = risk_assessment['compliance']
print(f"  • HIPAA Compliant: {'✓ YES' if compliance['hipaa_compliant'] else '⚠️  NO'}")
print(f"  • GDPR Compliant: {'✓ YES' if compliance['gdpr_compliant'] else '⚠️  NO'}")
print(f"  • Audit Trail Available: {'✓ YES' if compliance['audit_trail_available'] else '⚠️  NO'}")
print(f"\n  Recommendations:")
for rec in compliance['recommendations']:
    print(f"    → {rec}")

print("\n🚨 Overall Risk Level: " + risk_assessment['overall_risk_level'])

# Color coding
risk_colors = {
    'LOW': '🟢',
    'MEDIUM': '🟡',
    'HIGH': '🔴',
    'CRITICAL': '⛔'
}
risk_level = risk_assessment['overall_risk_level']
print(f"\n  {risk_colors.get(risk_level, '⚪')} {risk_level} RISK")

In [ ]:
print("="*60)
print("STRUCTURED OUTPUT GENERATION")
print("="*60)

# Combine all extraction results into structured format
structured_output = {
    'metadata': {
        'note_id': sample_clinical_notes[0]['note_id'],
        'extraction_timestamp': extraction_results['timestamp'],
        'extraction_method': 'Regex + spaCy + LLM Simulation',
        'pipeline_version': '1.0'
    },
    'clinical_data': {
        'icd_codes': extraction_results['regex_extraction']['icd_codes'],
        'dates': extraction_results['regex_extraction']['dates'],
        'lab_values': extraction_results['regex_extraction']['lab_values'],
        'dosages': extraction_results['regex_extraction']['dosages'],
        'clinical_concepts': extraction_results['clinical_concepts'],
        'llm_summary': {
            'chief_complaint': llm_results['chief_complaint'],
            'primary_diagnosis': llm_results['primary_diagnosis'],
            'secondary_diagnoses': llm_results['secondary_diagnoses']
        }
    },
    'risk_assessment': risk_assessment
}

# Display structured output
print("\n📊 Structured Clinical Data (JSON):")
print(json.dumps(structured_output, indent=2)[:1000] + "...")

# Create compliance report
compliance_report = f"""
╔════════════════════════════════════════════════════════════════╗
║           CLINICAL EXTRACTION COMPLIANCE REPORT                ║
╚════════════════════════════════════════════════════════════════╝

TIMESTAMP: {risk_assessment['timestamp']}
RISK LEVEL: {risk_assessment['overall_risk_level']}

DATA PRIVACY
───────────
PII Present:              {privacy['has_pii']}
PII Types Found:          {privacy['pii_types_found']}
Compliance Status:        {privacy['compliance_status']}
Action Required:          {privacy['recommendation']}

BIAS ASSESSMENT  
──────────────
Bias Detected:            {bias['potential_bias_detected']}
Demographic Terms:        {bias['demographic_terms_found']}
Human Review:             RECOMMENDED

REGULATORY COMPLIANCE
─────────────────────
HIPAA Compliant:          {compliance['hipaa_compliant']}
GDPR Compliant:           {compliance['gdpr_compliant']}
Audit Trail Available:    {compliance['audit_trail_available']}

AUDIT TRAIL (Last 5 Operations)
───────────────────────────────"""

for entry in compliance['audit_trail']:
    compliance_report += f"\n  Operation {entry['step']}: {entry['timestamp']}"

print(compliance_report)

# Summary statistics
print("\n\n📈 EXTRACTION SUMMARY STATISTICS")
print("─" * 60)
print(f"Total Entities Extracted:    {sum(len(v) for v in extraction_results.get('clinical_concepts', {}).values())}")
print(f"ICD Codes Found:             {len(extraction_results['regex_extraction']['icd_codes'])}")
print(f"Lab Values Detected:         {len(extraction_results['regex_extraction']['lab_values'])}")
print(f"Risk Alerts Generated:       {1 if risk_assessment['overall_risk_level'] != 'LOW' else 0}")

## 7. Generate Structured Output and Compliance Report

Export extraction results and risk assessment in standardized JSON format for downstream systems and compliance review.

## 6. Risk Assessment and Bias Detection

Comprehensive risk evaluation including data privacy, bias detection, HIPAA/GDPR compliance, and audit trails.

## 5. LLM-Based Extraction and Summarization

While full LLM integration requires API keys, here's a demonstration of how LLMs can augment extraction by:
- Normalizing extracted entities
- Providing contextual summaries
- Flagging clinical significance
- Generating risk alerts

## 4. spaCy NLP Tagging for Clinical Entities

Use spaCy to identify and tag medical entities like diagnoses, medications, and symptoms. 
**Note:** This requires spaCy and the en_core_web_sm model. If not available, we'll use keyword-based extraction instead.

## 3. Regex-Based Pattern Extraction

Extract structured data like dates, ICD codes, lab values, and dosages using regex patterns.

## 2. Load and Prepare Clinical Text Data

## 1. Import Required Libraries